In [1]:
import time
import numpy as np
import pandas as pd
from datetime import datetime
from tts_data_utils.core.data_frame import TtsDataFrame
from copy import deepcopy

In [2]:
class TestDataFrame(TtsDataFrame):
    SCHEMA = [
        ('label', str),
        ('value', (int, float, str)),
        ('units', str),
        ('time', datetime),
    ]

    TIME_FORMATS = {
        'time': '%Y-%jT%H:%M:%S.%f'
    }
    DEFAULT_TIME_LABEL = 'time'

    LABEL_COL = 'label'

    VALUE_COL = 'value'

    LABEL_COLUMN = 'label'

    DEFAULT_DIRECTION = 'horizontal'

    def __init__(self, *args, name=None, metadata=None, **kwargs):
        # Provide a sensible default name similar to EvrContainer
        if name is None:
            name = "Test DataFrame"
        super().__init__(*args, name=name, metadata=metadata, **kwargs)



## 1. Valid data — coerce + validate

In [3]:
df = TestDataFrame(
    data={
        "label":  ["temp",   "speed", "pressure", "speed"],
        "value":  [42,       "slow", 3.14,       "fast"],
        "units":  ["C",      "knots", "psi",      "knots"],
        "time":   ["2024-001T00:00:00.000000",
                   "2024-001T00:00:00.000000",
                   "2024-001T01:00:00.000000",
                   "2024-001T02:00:00.000000"],
    },
    coerce=True,
    validate=True,
)
display(df)
df.dtypes

,label,value,units,time
0,temp,42,C,2024-01-01 00:00:00
1,speed,slow,knots,2024-01-01 00:00:00
2,pressure,3.14,psi,2024-01-01 01:00:00
3,speed,fast,knots,2024-01-01 02:00:00


label               str
value            object
units               str
time     datetime64[us]
dtype: object

In [4]:
df.eq('value', 42)

,label,value,units,time
0,temp,42,C,2024-01-01


## 2. `.valid` property

In [5]:
df.valid

True

## 3. `'1.1'` string in int/float/str column — should land as float

In [6]:
df2 = TestDataFrame(
    data={
        "label":  ["x"],
        "value":  ["1.1"],
        "units":  ["m"],
        "time":   ["2024-001T00:00:00.000000"],
    },
    coerce=True,
    validate=True,
)
display(df2)
print("value[0]:", df2["value"].iloc[0], type(df2["value"].iloc[0]))

,label,value,units,time
0,x,1.1,m,2024-01-01


value[0]: 1.1 <class 'str'>


## 4. Validation failure — int in `units` column

In [7]:
try:
    df3 = TestDataFrame(
        data={
            "label":  ["y"],
            "value":  [1],
            "units":  [999],
            "time":   ["2024-001T00:00:00.000000"],
        },
        coerce=False,
        validate=True,
    )
except Exception as e:
    print("Caught expected error:", e)

Caught expected error: Column 'units' has values with invalid type for schema <class 'str'>. Example bad indices: [0]


## 5. Missing column

In [8]:
try:
    df4 = TestDataFrame(
        data={"label": ["a"], "value": [1], "units": ["m"]},
        coerce=True,
        validate=True,
    )
except Exception as e:
    print("Caught expected error:", e)

Caught expected error: Missing expected schema columns: ['time']; present columns: ['label', 'value', 'units']


## 6. Stress test — 10 M rows, 100 labels

In [9]:
N_ROWS        = 10_000
N_LABELS      = 100
TIME_START    = datetime(2024, 1, 1)
TIME_STEP_SEC = 1

label_names = [f"sensor_{i:03d}" for i in range(N_LABELS)]
units_map   = {l: f"unit_{i % 10}" for i, l in enumerate(label_names)}

rng        = np.random.default_rng(42)
base_times = pd.date_range(TIME_START, periods=N_ROWS // N_LABELS, freq=f"{TIME_STEP_SEC}s")
# Each label gets exactly one row per timestamp; shuffle rows for realism
labels = np.repeat(label_names, N_ROWS // N_LABELS)
times  = np.tile(base_times, N_LABELS)
idx    = rng.permutation(N_ROWS)
labels, times = labels[idx], times[idx]
units  = np.array([units_map[l] for l in labels])
values = rng.random(N_ROWS)

t0 = time.perf_counter()
stress_df = TestDataFrame(
    data={"label": labels, "value": values, "units": units, "time": times},
    coerce=False,
    validate=False,
)
print(f"Built in {time.perf_counter() - t0:.2f}s")

# Thin out sensor_001 (every 3rd sample) and sensor_002 (every 5th sample)
# so they have gaps relative to the other channels.
drop_001 = stress_df.index[stress_df['label'] == 'sensor_001'][::3]
drop_002 = stress_df.index[stress_df['label'] == 'sensor_002'][::5]
stress_df = stress_df.drop(index=drop_001.union(drop_002))

Built in 0.00s


In [10]:
stress_df[['time', 'value']].info(memory_usage='deep')

<class '__main__.TestDataFrame'>
Index: 9946 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   time    9946 non-null   datetime64[us]
 1   value   9946 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 233.1 KB


## 7. Pivot to wide form (with cache)

In [11]:
t1 = time.perf_counter()
wide = stress_df.wide
print(f"First pivot:  {time.perf_counter() - t1:.2f}s  shape={wide.shape}")

t2 = time.perf_counter()
wide = stress_df.wide
print(f"Cached pivot: {time.perf_counter() - t2:.4f}s  shape={wide.shape}")
#stress_df._pivot_cache.info(memory_usage='deep')

First pivot:  0.01s  shape=(100, 100)
Cached pivot: 0.0017s  shape=(100, 100)


## 8. derive_values — sensor_001 * 2 - abs(sensor_002)

In [14]:
t3 = time.perf_counter()
from datetime import timedelta
derived = stress_df.derive_values('new_channel = sensor_002 < 0.5 and sensor_001 > 0.1', timeout=timedelta(seconds=0))
print(f"derive_values: {time.perf_counter() - t3:.2f}s")
display(derived.head(10))

derive_values: 0.02s


,time,label,value
0,2024-01-01 00:00:00,new_channel,False
1,2024-01-01 00:00:01,new_channel,False
2,2024-01-01 00:00:02,new_channel,True
3,2024-01-01 00:00:03,new_channel,True
4,2024-01-01 00:00:04,new_channel,False
5,2024-01-01 00:00:11,new_channel,True
6,2024-01-01 00:00:12,new_channel,True
7,2024-01-01 00:00:13,new_channel,False
8,2024-01-01 00:00:14,new_channel,True
9,2024-01-01 00:00:15,new_channel,False


In [ ]:
len(stress_df.query('label == "sensor_001"'))

In [ ]:
512*60*60*24/1e6

In [ ]:
len(derived)

In [ ]:
lc = stress_df.LABEL_COL
tc = stress_df.DEFAULT_TIME_LABEL
s1 = set(stress_df.loc[stress_df[lc] == 'sensor_001', tc])
s2 = set(stress_df.loc[stress_df[lc] == 'sensor_002', tc])
s5 = set(stress_df.loc[stress_df[lc] == 'sensor_005', tc])
print(len(s1 & s2 & s5))


In [ ]:
import pandas as pd
a = pd.DataFrame({'a':[1,2,3], 'b': [0,2,3]})
b = pd.DataFrame({'a':[3,2,1], 'b': [3,2,0]})

In [ ]:
set(a['a']) & set(a['b'])

In [ ]:
import pandas as pd

# Two DataFrames with different columns and row indices
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]}, index=[0, 1])
df2 = pd.DataFrame({'A': [1, 5], 'C': [6, 7]}, index=[0, 2])

# 1. Align them using an outer join to equalize their shapes
df1_aligned, df2_aligned = df1.align(df2, join='outer')

# 2. Run the standard comparison
diff = df1_aligned.compare(df2_aligned)
print(diff)

In [ ]:
import datacompy

# It requires a unique key column (or set of columns) to match rows together
compare = datacompy.PandasCompare(
    a.reset_index(), 
    a.reset_index(), 
    join_columns='index'
)

# Generates a comprehensive human-readable summary report
print(compare.matches())

In [ ]:
df

In [ ]:
stress_df._data_hash = None
asdf = stress_df.timer().wide
asdf = stress_df.timer().wide

In [ ]:
asdf = stress_df.timer().at_times_where('sensor_000 > 0.5')

In [ ]:
from datetime import datetime
from tts_data_utils.core.data_frame import TtsDataFrame

class EventFrame(TtsDataFrame):
    SUBCONTAINER_KEY = 'event_id'

class DetailFrame(TtsDataFrame):
    pass

parent = EventFrame({
    'event_id': [f'EVT-{i:03d}' for i in range(10)],
    'name':     ['alpha','beta','gamma','delta','epsilon',
                 'zeta','eta','theta','iota','kappa'],
    'value':    [10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
})

for eid, name in zip(parent['event_id'], parent['name']):
    child = DetailFrame({
        'param':   ['x', 'y', 'z'],
        'reading': [round(hash(eid + p) % 100 / 10, 2) for p in ['x','y','z']],
        'note':    [f'{name} param {p}' for p in ['x','y','z']],
    })
    parent.set_subcontainer(eid, 'details', child)

parent._subcontainers['EVT-001']['details']
